# PICasso Layout Visualization Demo

This notebook demonstrates how to visualize photonic circuit layouts generated by the PICasso framework.

## Features:
- Load and display GDS files
- Inline visualization with `circuit.plot()`
- Side-by-side comparison of designs
- High-resolution exports
- Integration with validation results
y

In [1]:
# Core imports
import gdsfactory as gf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

print(f"✅ GDSFactory version: {gf.__version__}")


✅ GDSFactory version: 8.32.2


## 1. Basic Layout Visualization - Creating a Simple MZM


In [2]:
# Create a simple Mach-Zehnder Modulator
circuit = gf.Component("demo_mzm")

# Add splitter
splitter = circuit.add_ref(gf.components.mmi1x2())
splitter.move((0, 0))

# Add combiner (mirrored)
combiner = circuit.add_ref(gf.components.mmi2x1())
combiner.mirror()
combiner.move((250, 0))

# Add phase shifters
ps1 = circuit.add_ref(gf.components.straight_heater_metal(length=50))
ps1.move((100, 50))

ps2 = circuit.add_ref(gf.components.straight_heater_metal(length=50))
ps2.move((100, -50))

# Route connections
gf.routing.route_bundle(
    circuit,
    [splitter.ports['o2'], splitter.ports['o3']],
    [ps1.ports['o1'], ps2.ports['o1']],
    cross_section='strip',
    radius=15,
    separation=15
)

gf.routing.route_bundle(
    circuit,
    [ps1.ports['o2'], ps2.ports['o2']],
    [combiner.ports['o2'], combiner.ports['o1']],
    cross_section='strip',
    radius=15,
    separation=15
)

# Add external ports
circuit.add_port('o1', port=splitter.ports['o1'])
circuit.add_port('o2', port=combiner.ports['o3'])

# Visualize
circuit.draw_ports()
circuit.plot()


AttributeError: module 'gdsfactory.components' has no attribute 'mmi2x1'

## 2. Load and Display Existing GDS Files from Results


In [ ]:
# Load results CSV
results_dir = Path('hf_inference_workflow/output/results')
gds_dir = Path('hf_inference_workflow/output/gds_files')

# List available result files
csv_files = list(results_dir.glob('*.csv'))
print(f"Found {len(csv_files)} result files")

# Load latest framework results
if (results_dir / 'framework_results.csv').exists():
    df = pd.read_csv(results_dir / 'framework_results.csv')
    print(f"✅ Loaded {len(df)} designs from framework_results.csv")
    print(f"   Success rate: {df['success'].mean()*100:.1f}%")
    
    # Display first few successful designs
    successful = df[df['success'] == True]
    if len(successful) > 0:
        for idx, row in successful.head(2).iterrows():
            problem_idx = row['problem_idx']
            sample_idx = row['sample_idx']
            circuit_type = row['circuit_type']
            
            print(f"\n{'='*50}")
            print(f"Problem {problem_idx}: {circuit_type} (Sample {sample_idx})")
            print(f"{'='*50}")
            
            # Find GDS file
            gds_files = list(gds_dir.glob(f"problem_{problem_idx}_sample_{sample_idx}_*.gds"))
            if gds_files:
                print(f"Loading: {gds_files[0].name}")
                try:
                    component = gf.import_gds(gds_files[0])
                    component.plot()
                    plt.title(f"{circuit_type} - Problem {problem_idx}, Sample {sample_idx}")
                    plt.show()
                except Exception as e:
                    print(f"Error loading GDS: {e}")
else:
    print("⚠️ No framework_results.csv found. Run gen_data_validated.py first.")
